**Set environment**

In [1]:
source ../run_config_project.sh
show_env

BASE DIRECTORY (FD_BASE):      /hpc/group/igvf/kk319
REPO DIRECTORY (FD_REPO):      /hpc/group/igvf/kk319/repo
WORK DIRECTORY (FD_WORK):      /hpc/group/igvf/kk319/work
DATA DIRECTORY (FD_DATA):      /hpc/group/igvf/kk319/data
CONTAINER DIR. (FD_SING):      /hpc/group/igvf/kk319/container

You are working with           
PATH OF PROJECT (FD_PRJ):      /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR
PROJECT RESULTS (FD_RES):      /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/results
PROJECT SCRIPTS (FD_EXE):      /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/scripts
PROJECT DATA    (FD_DAT):      /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/data
PROJECT NOTE    (FD_NBK):      /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/notebooks
PROJECT DOCS    (FD_DOC):      /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/docs
PROJECT LOG     (FD_LOG):      /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/log
PROJECT REF     (FD_REF):      /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/references
PR

## Preview

In [2]:
ls -1 ${FD_RES}

analysis_variant_motif_richard
analysis_variant_motif_richard_arc251231
predict_variant_alphagenome
predict_variant_kircher2019


In [3]:
ls ${FD_RES}/analysis_variant_motif_richard

background_zero_order.npy
background_zero_order.tsv
batches_dev
batches_pilot
batches_top
motifdelta_pilot_jaspar2024
motifdelta_top_jaspar2024
motifdelta_top_jvierstra_v2.1beta
motif_jaspar2024_core_vertebrates_nonredundant.lods.pkl
motif_jaspar2024_core_vertebrates_nonredundant.pmap.pkl
motif_jaspar2024_core_vertebrates_nonredundant.tbind.pkl
motif_nonredundant_jvierstra_v2.1beta.lods.pkl
motif_nonredundant_jvierstra_v2.1beta.pmap.pkl
motif_nonredundant_jvierstra_v2.1beta.tbind.pkl
motifscan_pilot_jaspar2024
motifscan_top_jaspar2024
motifscan_top_jvierstra_v2.1beta
variant_closed_gof_bluestarr.flankL35R70.ref.fa
variant_closed_gof_bluestarr.tsv


In [4]:
ls ${FD_RES}/analysis_variant_motif_richard/batches_pilot

variant_closed_gof_bluestarr.flankL35R70.pilot_chunk001.ref.fa.gz
variant_closed_gof_bluestarr.flankL35R70.pilot_chunk002.ref.fa.gz
variant_closed_gof_bluestarr.flankL35R70.pilot_chunk003.ref.fa.gz
variant_closed_gof_bluestarr.flankL35R70.pilot_chunk004.ref.fa.gz
variant_closed_gof_bluestarr.flankL35R70.pilot_chunk005.ref.fa.gz
variant_closed_gof_bluestarr.flankL35R70.pilot_chunk006.ref.fa.gz
variant_closed_gof_bluestarr.flankL35R70.pilot_chunk007.ref.fa.gz
variant_closed_gof_bluestarr.flankL35R70.pilot_chunk008.ref.fa.gz
variant_closed_gof_bluestarr.flankL35R70.pilot_chunk009.ref.fa.gz
variant_closed_gof_bluestarr.flankL35R70.pilot_chunk010.ref.fa.gz
variant_closed_gof_bluestarr.flankL35R70.pilot_chunk011.ref.fa.gz
variant_closed_gof_bluestarr.flankL35R70.pilot_chunk012.ref.fa.gz
variant_closed_gof_bluestarr.flankL35R70.pilot_chunk013.ref.fa.gz
variant_closed_gof_bluestarr.flankL35R70.pilot_chunk014.ref.fa.gz
variant_closed_gof_bluestarr.flankL35R70.pilot_chunk015.ref.fa.gz
variant_cl

## Non-redundant motifs

### Prepare

In [5]:
FD_BATCH=${FD_RES}/analysis_variant_motif_richard/batches_pilot
FD_MSCAN=${FD_RES}/analysis_variant_motif_richard/motifscan_pilot_jvierstra_v2.1beta
FD_DELTA=${FD_RES}/analysis_variant_motif_richard/motifdelta_pilot_jvierstra_v2.1beta

FP_MOTIF=${FD_RES}/analysis_variant_motif_richard/motif_nonredundant_jvierstra_v2.1beta.lods.pkl
FP_TBIND=${FD_RES}/analysis_variant_motif_richard/motif_nonredundant_jvierstra_v2.1beta.tbind.pkl

mkdir -p "${FD_MSCAN}"
mkdir -p "${FD_DELTA}"

### Execute

In [31]:
### set script
FP_EXE=${FD_EXE}/run_motifdelta_01_scan.sh

### set slurm opt
NUM_CPU=2
NUM_MEM=50G

LST_OPTS=(
    -A majoroslab
    -p igvf,common
    --exclude=dcc-comp-10,dcc-core-08,dcc-core-53
    --cpus-per-task="${NUM_CPU}"
    --mem="${NUM_MEM}"
    --chdir="${FD_EXE}"
    --export=ALL,FP_CNF="${FP_CNF}"
    --parsable
)

### loop init
LST_JOBS=()
FN_PREFIX=variant_closed_gof_bluestarr.flankL35R70

### Loop through I/O
for i in $(seq -w 001 020); do
    
    ### set I/O
    TXT_TAG="pilot_chunk${i}"
    TXT_JOB="motifscan.jvierstra.${TXT_TAG}"
    
    FP_INP=${FD_BATCH}/${FN_PREFIX}.${TXT_TAG}.ref.fa.gz
    FP_OUT=${FD_MSCAN}/${FN_PREFIX}.${TXT_TAG}.npz
    FP_LOG=${FD_LOG}/run.${TXT_JOB}.txt
    #FP_LOG=${FD_LOG}/run.motifscan.jaspar.batch.${TXT_TAG}.%j.txt
    
    NUM_FLANK_LEFT=35
    NUM_BATCH_SIZE=2000

    ### check file existence
    if [[ ! -f "${FP_INP}" ]]; then
        echo "Missing input: ${FP_INP}"
        continue
    fi
    
    ### execute
    JOBID=$(sbatch   \
        "${LST_OPTS[@]}" \
        --job-name="${TXT_JOB}" \
        --output="${FP_LOG}"    \
        "${FP_EXE}" "${FP_INP}" "${FP_MOTIF}" "${FP_OUT}" "${NUM_FLANK_LEFT}" "${NUM_BATCH_SIZE}"
    )
    echo "Submitted ${TXT_TAG}: ${JOBID}"
    LST_JOBS+=("${JOBID}")
done

Submitted pilot_chunk001: 42208055
Submitted pilot_chunk002: 42208056
Submitted pilot_chunk003: 42208057
Submitted pilot_chunk004: 42208058
Submitted pilot_chunk005: 42208059
Submitted pilot_chunk006: 42208060
Submitted pilot_chunk007: 42208061
Submitted pilot_chunk008: 42208062
Submitted pilot_chunk009: 42208063
Submitted pilot_chunk010: 42208064
Submitted pilot_chunk011: 42208065
Submitted pilot_chunk012: 42208066
Submitted pilot_chunk013: 42208067
Submitted pilot_chunk014: 42208068
Submitted pilot_chunk015: 42208069
Submitted pilot_chunk016: 42208070
Submitted pilot_chunk017: 42208071
Submitted pilot_chunk018: 42208072
Submitted pilot_chunk019: 42208073
Submitted pilot_chunk020: 42208074


## Review

In [35]:
sacct_summary.sh "${LST_JOBS[@]}"

Detected multiple job IDs (20). Running batch summary.
===== Summary (.ba tasks) =====
JobID                               JobName      State ExitCode ElapsedRaw   TotalCPU     MaxRSS                       NodeList 
------------ ------------------------------ ---------- -------- ---------- ---------- ---------- ------------------------------ 
42208055.ba+                          batch  COMPLETED      0:0         49  00:21.800  11984516K                    dcc-comp-07 
42208056.ba+                          batch  COMPLETED      0:0        127  00:52.526  11966644K                    dcc-core-50 
42208057.ba+                          batch  COMPLETED      0:0        126  00:53.045  11967084K                    dcc-core-50 
42208058.ba+                          batch  COMPLETED      0:0        109  00:47.716  11959300K                    dcc-core-50 
42208059.ba+                          batch  COMPLETED      0:0        125  00:52.835  11967728K                    dcc-core-50 
42208060.b

In [36]:
cat ${FD_LOG}/run.motifscan.jvierstra.pilot_chunk001.txt

Hostname:           dcc-comp-07
Slurm Array Index:  NA
Time Stamp:         01-19-26+15:14:06
PWD:                /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/scripts

Loading FASTA sequences...
Loaded 5000 sequences
Load and check complete in 0.06 seconds

Loading motif matrices...
Loaded 637 motifs from /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/results/analysis_variant_motif_richard/motif_nonredundant_jvierstra_v2.1beta.lods.pkl
Load and check complete in 0.01 seconds

Setting motif kernel...
Forward kernels shape: (637, 28, 4)
Reverse kernels shape: (637, 28, 4)
Set complete in 0.00 seconds

Running motif scanning...
Scan complete in 9.17 seconds
Output array size (ref+obs+unobs): 5.624 GB

Saving results...
Saved results to /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/results/analysis_variant_motif_richard/motifscan_pilot_jvierstra_v2.1beta/variant_closed_gof_bluestarr.flankL35R70.pilot_chunk001.npz
Saved complete in 29.18 seconds


Done!
Run Time: 39 seconds



In [37]:
cat ${FD_LOG}/run.motifscan.jvierstra.pilot_chunk020.txt

Hostname:           dcc-core-07
Slurm Array Index:  NA
Time Stamp:         01-19-26+15:15:41
PWD:                /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/scripts

Loading FASTA sequences...
Loaded 5000 sequences
Load and check complete in 0.05 seconds

Loading motif matrices...
Loaded 637 motifs from /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/results/analysis_variant_motif_richard/motif_nonredundant_jvierstra_v2.1beta.lods.pkl
Load and check complete in 0.01 seconds

Setting motif kernel...
Forward kernels shape: (637, 28, 4)
Reverse kernels shape: (637, 28, 4)
Set complete in 0.01 seconds

Running motif scanning...
Scan complete in 43.47 seconds
Output array size (ref+obs+unobs): 5.624 GB

Saving results...
Saved results to /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/results/analysis_variant_motif_richard/motifscan_pilot_jvierstra_v2.1beta/variant_closed_gof_bluestarr.flankL35R70.pilot_chunk020.npz
Saved complete in 82.27 seconds


Done!
Run Time: 2 minutes and 7 seconds
